# TopicVisExplorer — Paper-Faithful Pipeline: Single & Multi-Corpus

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezf/TopicVisExplorer/blob/main/examples/07_single_and_multicorpus.ipynb)

> **Colab users:** run `!pip install git+https://github.com/gonzalezf/TopicVisExplorer.git` in the first cell.

This notebook demonstrates the **paper-faithful default pipeline** end-to-end:

1. **Text preprocessing** with `text_cleaner_batch` — NLTK tokenisation + spaCy lemmatisation (NOUN/ADJ/VERB/ADV only)
2. **Topic modelling** with Gensim LDA via `GensimLDAAdapter`
3. **Word2Vec embeddings** (`Word2Vec.fit`) at the paper's default hyper-parameters (300-dim CBOW)
4. **Coherence metrics** via `coherence_report`
5. **Static HTML export** with `tve.save_html` (non-blocking — safe inside a notebook)

**Part 2** repeats the same pipeline for two separate corpora and explains how to launch the
Sankey (multi-corpus comparison) view.

---

### Prerequisites

```bash
# from the repo root
uv sync --all-extras
uv run python -m spacy download en_core_web_sm
uv run python -c "import nltk; nltk.download('stopwords')"
```

All cells use the **non-blocking** `tve.prepare()` + `tve.save_html()` path.
`tve.show()` starts a long-running server — see the terminal command at the end of each part.

In [1]:
# Guard: verify [full] extras are installed before running any heavy cells
import importlib

_missing = []
for pkg in ("spacy", "nltk"):
    if importlib.util.find_spec(pkg) is None:
        _missing.append(pkg)

if _missing:
    raise ImportError(
        f"Missing packages: {_missing}.\n"
        "Install the [full] extra:\n"
        "  uv sync --all-extras\n"
        "  uv run python -m spacy download en_core_web_sm\n"
        "  uv run python -c \"import nltk; nltk.download('stopwords')\""
    )

import topicvisexplorer as tve  # noqa: E402

print(f"topicvisexplorer {tve.__version__} — [full] extras present")

topicvisexplorer 1.0.0 — [full] extras present


---
## Part 1 — Single Corpus (paper-style defaults)

We use four 20 Newsgroups categories as a single mixed corpus so the topics are
meaningfully diverse: *space*, *medicine*, *baseball*, and *politics*.

In [2]:
from sklearn.datasets import fetch_20newsgroups

CATEGORIES = ["sci.space", "sci.med", "rec.sport.baseball", "talk.politics.misc"]

raw = fetch_20newsgroups(
    subset="train",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    random_state=42,
)

# Keep non-empty documents only
raw_texts = [t.strip() for t in raw.data if t.strip()]
print(f"Loaded {len(raw_texts)} documents across {len(CATEGORIES)} categories")
print(f"Example (first 200 chars):\n  {raw_texts[0][:200]!r}")

Loaded 2183 documents across 4 categories
Example (first 200 chars):
  "Bingo.\n Nothing evil at all. There's no actual harm in what they're doing, only\nhow they represent it.\n\n -----------------------------------------------------------------\n .sig files are like strings "


### Step 1 — Text preprocessing (paper-faithful)

`text_cleaner_batch` applies:
- Unicode normalisation + diacritics/digit stripping
- NLTK TweetTokenizer + stopword removal
- spaCy `en_core_web_sm` lemmatisation (NOUN / ADJ / VERB / ADV only)

Output: `list[list[str]]` — one token list per document.

In [3]:
from topicvisexplorer.preprocessing import text_cleaner_batch

print("Preprocessing …")
tokenized = text_cleaner_batch(raw_texts)

# Drop very short documents (< 5 tokens) — too sparse for LDA
pairs = [(t, r) for t, r in zip(tokenized, raw_texts, strict=False) if len(t) >= 5]
tokenized, raw_texts = [p[0] for p in pairs], [p[1] for p in pairs]

print(f"After filtering: {len(tokenized)} documents")
print(f"Vocabulary sample (doc 0): {tokenized[0][:15]}")

Preprocessing …


After filtering: 2089 documents
Vocabulary sample (doc 0): ['evil', 'actual', 'harm', 'represent', 'sig', 'file', 'string', 'get']


### Step 2 — Fit Gensim LDA

In [4]:
from gensim.corpora import Dictionary
from gensim.models import LdaModel

from topicvisexplorer.models import GensimLDAAdapter

NUM_TOPICS = 5
RANDOM_STATE = 42

dictionary = Dictionary(tokenized)
dictionary.filter_extremes(no_below=5, no_above=0.8)
corpus = [dictionary.doc2bow(doc) for doc in tokenized]

lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=RANDOM_STATE,
    passes=15,
)

adapter = GensimLDAAdapter()
model_data = adapter.extract(lda, corpus=corpus, dictionary=dictionary)

prepared = tve.prepare(
    topic_term_dists=model_data.topic_term_dists,
    doc_topic_dists=model_data.doc_topic_dists,
    doc_lengths=model_data.doc_lengths,
    vocab=model_data.vocab,
    term_frequency=model_data.term_frequency,
)

K, V = model_data.topic_term_dists.shape
N = model_data.doc_topic_dists.shape[0]
print(f"LDA fitted: K={K} topics, V={V} vocab terms, N={N} documents")

LDA fitted: K=5 topics, V=3849 vocab terms, N=2089 documents


### Step 3 — Fit Word2Vec (paper defaults)

Paper hyper-parameters: 300-dim CBOW, window=5, min_count=5, 50 epochs, seed=42.
Requires ≥ 200 documents.

In [5]:
from topicvisexplorer.embeddings import Word2Vec

print("Training Word2Vec …")
w2v = Word2Vec.fit(tokenized)  # uses paper defaults

print(repr(w2v))  # e.g. Word2Vec(vocab=3210, vector_size=300)
# Show the vector for one known in-vocabulary term
top_term = prepared.topic_top_terms(prepared.topic_order[0], n=1)[0]
if top_term in w2v:
    print(f"Vector shape for \'{top_term}\': {w2v[top_term].shape}")
    print(f"First 5 values: {w2v[top_term][:5]}")
else:
    print(f"\'{top_term}\' is OOV (min_count=5 filter) — this is expected for rare terms")


Training Word2Vec …


Word2Vec(vocab=4,614, vector_size=300)
Vector shape for 'space': (300,)
First 5 values: [1.8586075  0.8848446  1.794788   0.75820524 0.68091816]


In [6]:
from topicvisexplorer.coherence import report as coherence_report

print("Top-5 terms per topic (lambda=0.6):\n")
for topic_id in prepared.topic_order:
    terms = prepared.topic_top_terms(topic_id, n=5)
    print(f"  Topic {topic_id}: {', '.join(terms)}")

print()
coh = coherence_report(
    prepared,
    tokenized_texts=tokenized,
    doc_topic_dists=model_data.doc_topic_dists,
)
print(coh)

Top-5 terms per topic (lambda=0.6):

  Topic 1: space, system, design, launch, station
  Topic 2: food, police, search, say, file
  Topic 3: go, good, year, make, think
  Topic 4: team, run, drug, catcher, help
  Topic 5: pain, doctor, take, sinus, patient

CoherenceReport(npmi=[0.24698146300330764, 0.21854579352188322, 0.20239906885723077, 0.17796824508457873, 0.299361376628424], c_v=[0.7245394609037834, 0.7117863327173074, 0.7549193992258664, 0.660928058228839, 0.8421624475688873], segregation=[1.0, 0.9868421052631579, 1.0, 0.9868421052631579, 1.0], coverage=[0.3130684538056486, 0.059837242699856394, 0.3820009573958832, 0.12206797510770703, 0.12302537099090474], labels=['space system design', 'food police search', 'go good year', 'team run drug', 'pain doctor take'])


In [7]:
from pathlib import Path

out_html = Path("single_corpus_output.html")
tve.save_html(prepared, str(out_html))
print(f"Saved static HTML → {out_html.resolve()}  ({out_html.stat().st_size / 1024:.0f} KB)")
print()
print("Open in your browser:")
print(f"  file://{out_html.resolve()}")


Saved static HTML → /root/projects/topicvisexplorer-lib/examples/single_corpus_output.html  (2139 KB)

Open in your browser:
  file:///root/projects/topicvisexplorer-lib/examples/single_corpus_output.html


### Launch the interactive browser UI

`tve.show()` starts a **blocking** FastAPI server — run it from a terminal, not from inside
a notebook cell:

```python
import topicvisexplorer as tve
tve.show(prepared, raw_texts=raw_texts, model_data=model_data, embedding=w2v)
```

This opens the LDAvis-style interactive panel at `http://localhost:8000`.
The **Omega slider** in the UI uses the Word2Vec embedding to reorder terms.
See [quickstart.md](../docs/quickstart.md) for all `tve.show()` options.

---
## Part 2 — Multi-Corpus Sankey (paper-style defaults)

We build **two separate corpora** — science vs. politics — each fitted independently.
Passing both `PreparedData` objects to `tve.show()` activates the Sankey view that shows
how topics from Corpus A map onto topics from Corpus B.

- **Corpus A:** `sci.space` + `sci.med`
- **Corpus B:** `talk.politics.guns` + `talk.politics.misc`

In [8]:
# Load and preprocess both corpora
raw_a = fetch_20newsgroups(
    subset="train",
    categories=["sci.space", "sci.med"],
    remove=("headers", "footers", "quotes"),
    random_state=42,
)
raw_b = fetch_20newsgroups(
    subset="train",
    categories=["talk.politics.guns", "talk.politics.misc"],
    remove=("headers", "footers", "quotes"),
    random_state=42,
)

texts_a = [t.strip() for t in raw_a.data if t.strip()]
texts_b = [t.strip() for t in raw_b.data if t.strip()]

print("Preprocessing corpus A …")
tok_a = text_cleaner_batch(texts_a)
print("Preprocessing corpus B …")
tok_b = text_cleaner_batch(texts_b)

# Filter very short docs
pairs_a = [(t, r) for t, r in zip(tok_a, texts_a, strict=False) if len(t) >= 5]
pairs_b = [(t, r) for t, r in zip(tok_b, texts_b, strict=False) if len(t) >= 5]
tok_a, texts_a = [p[0] for p in pairs_a], [p[1] for p in pairs_a]
tok_b, texts_b = [p[0] for p in pairs_b], [p[1] for p in pairs_b]

print(f"Corpus A (science):  {len(tok_a)} docs")
print(f"Corpus B (politics): {len(tok_b)} docs")

Preprocessing corpus A …


Preprocessing corpus B …


Corpus A (science):  1126 docs
Corpus B (politics): 952 docs


In [9]:
def _fit_lda_prepared(tokenized_texts, num_topics=4, passes=15, seed=42):
    """Fit Gensim LDA and return (model_data, prepared)."""
    dct = Dictionary(tokenized_texts)
    dct.filter_extremes(no_below=5, no_above=0.8)
    corp = [dct.doc2bow(doc) for doc in tokenized_texts]
    lda_m = LdaModel(corpus=corp, id2word=dct, num_topics=num_topics,
                     random_state=seed, passes=passes)
    md = GensimLDAAdapter().extract(lda_m, corpus=corp, dictionary=dct)
    prep = tve.prepare(
        topic_term_dists=md.topic_term_dists,
        doc_topic_dists=md.doc_topic_dists,
        doc_lengths=md.doc_lengths,
        vocab=md.vocab,
        term_frequency=md.term_frequency,
    )
    return md, prep


print("Fitting LDA on corpus A …")
md_a, prep_a = _fit_lda_prepared(tok_a)
print(f"  Corpus A: K={md_a.topic_term_dists.shape[0]}, V={md_a.topic_term_dists.shape[1]}")

print("Fitting LDA on corpus B …")
md_b, prep_b = _fit_lda_prepared(tok_b)
print(f"  Corpus B: K={md_b.topic_term_dists.shape[0]}, V={md_b.topic_term_dists.shape[1]}")

# Save both PreparedData objects so you can launch the browser without refitting
prep_a.save("prep_science.npz")
prep_b.save("prep_politics.npz")
print()
print("Saved prep_science.npz and prep_politics.npz")
print("Launch the Sankey view from a terminal (see markdown cell below).")


Fitting LDA on corpus A …


  Corpus A: K=4, V=2714
Fitting LDA on corpus B …


  Corpus B: K=4, V=2453

Saved prep_science.npz and prep_politics.npz
Launch the Sankey view from a terminal (see markdown cell below).


In [10]:
# Compare top-3 terms per topic side by side
print("Corpus A — Science topics:")
for tid in prep_a.topic_order:
    print(f"  Topic {tid}: {', '.join(prep_a.topic_top_terms(tid, n=3))}")

print()
print("Corpus B — Politics topics:")
for tid in prep_b.topic_order:
    print(f"  Topic {tid}: {', '.join(prep_b.topic_top_terms(tid, n=3))}")

Corpus A — Science topics:
  Topic 1: food, msg, doctor
  Topic 2: use, year, station
  Topic 3: get, water, soon
  Topic 4: space, launch, satellite

Corpus B — Politics topics:
  Topic 1: gun, firearm, weapon
  Topic 2: people, program, work
  Topic 3: say, go, president
  Topic 4: people, government, right


### Launch the Sankey view

The multi-corpus Sankey requires a running server — `tve.show()` is **blocking** and cannot run inside a notebook cell. Run this from a terminal after the notebook completes:

```python
import topicvisexplorer as tve

# Load the fitted PreparedData objects saved above (no refitting needed)
prep_a = tve.load("prep_science.npz")
prep_b = tve.load("prep_politics.npz")

tve.show(
    [prep_a, prep_b],
    scenario_name="science_vs_politics",
)
```

This opens the **Sankey view** at `http://localhost:8000`. The diagram shows directed arrows from each science topic to the politics topic it overlaps with most in vocabulary space. Thick arrows = high overlap; no arrow = corpus-specific topic.

Stop the server with **Ctrl+C**.

See [quickstart.md](../docs/quickstart.md) for all `tve.show()` options.
